In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
import requests

df_fraud  = spark.sql("SELECT * FROM catalog1.gold_layer.gold_fraud_table")
df_claim  = spark.sql("SELECT * FROM catalog1.gold_layer.gold_claims_table")
df_policy = spark.sql("SELECT * FROM catalog1.gold_layer.gold_policy_table")

def table_to_text(df, table_name):
    rows = df.collect()
    columns = df.columns
    data = "\n".join([", ".join([f"{col}: {row[col]}" for col in columns]) for row in rows])
    return f"=== {table_name} ===\n{data}"

fraud_data  = table_to_text(df_fraud,  "Claims & Diagnosis Data")
claim_data  = table_to_text(df_claim,  "Fraud Analysis & Overpricing Data")
policy_data = table_to_text(df_policy, "Policy Coverage Data")

my_data = f"{fraud_data}\n\n{claim_data}\n\n{policy_data}"

print(f"Fraud table:  {df_fraud.count()} rows")
print(f"Claim table:  {df_claim.count()} rows")
print(f"Policy table: {df_policy.count()} rows")

In [0]:
import requests

COHERE_API_KEY = os.getenv("COHERE_API_KEY") 

def chat(user_question):
    response = requests.post(
        "https://api.cohere.com/v2/chat",
        headers={
            "Authorization": f"Bearer {COHERE_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "command-a-03-2025",
            "messages": [
                {
                    "role": "system",
                    "content": f"""You are a helpful insurance claims assistant.
                                You have access to 3 tables:
                                1. Claims & Diagnosis Data - contains claim details, patient, procedure costs
                                2. Fraud Analysis & Overpricing Data - contains fraud flags, fraud reasons, overpricing %
                                3. Policy Coverage Data - contains policy limits, coverage details

                                Answer questions in a clear and conversational way based ONLY on the data below.
                                If a question needs data from multiple tables, combine them using claim_id or policy_name.
                                If answer is not in data, say 'I could not find that information'.

                                Data:
                                {my_data}
                                """
                },
                {"role": "user", "content": user_question}
            ],
            "max_tokens": 300,
            "temperature": 0.3
        }
    )

    if response.status_code == 200:
        result = response.json()
        print(f"\nYou: {user_question}")
        print(f"HI THERE...! HERE IS YOUR INFORMATION :- {result['message']['content'][0]['text'].strip()}")
        print("-" * 50)
    else:
        print("Error:", response.status_code, response.text)

In [0]:
while(True):
    x= input("ASK Anything Iam your assistant.....!")
    chat(x)
    y=input("if yout want to exit give Yes otherwise No")
    if(y=="Yes"):
        break
    elif(y=="No"):
        continue